In [5]:
"""
run_audit.ipynb

Run the 50 selected Male patient prompts through GPT-4o-mini
using Azure OpenAI and save all treatment responses to a CSV.

Experimental design:
- Gender: Male
- Age: [40-50)
- Race: AfricanAmerican and Caucasian
- Pain levels: 0, 2, 4, 6, 8
- 5 patients per race for each pain level
- Total: 50 prompts
"""

# ==========================================
# 0. Import packages
# ==========================================

import pandas as pd
import time
import os

from dotenv import load_dotenv
from openai import OpenAI


# ==========================================
# 1. Load Azure information from key.env
# ==========================================

load_dotenv("key.env")

AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_API_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")


# Check Azure information
if not AZURE_ENDPOINT:
    raise ValueError("AZURE_ENDPOINT was not found in key.env")

if not AZURE_API_KEY:
    raise ValueError("AZURE_API_KEY was not found in key.env")

if not DEPLOYMENT_NAME:
    raise ValueError("DEPLOYMENT_NAME was not found in key.env")


# ==========================================
# 2. Input / Output files
# ==========================================

# 50 selected Male prompts
INPUT_CSV = "data_exp/male_50_prompts.csv"

# GPT responses will be saved here
OUTPUT_CSV = "results/responses/male_50_responses.csv"

TEMPERATURE = 0.1


# ==========================================
# 3. Run audit
# ==========================================

def run_audit():

    # --------------------------------------
    # Connect to Azure OpenAI
    # --------------------------------------

    client = OpenAI(
        base_url=AZURE_ENDPOINT.rstrip("/") + "/openai/v1/",
        api_key=AZURE_API_KEY,
    )


    # --------------------------------------
    # Read the 50 prompts
    # --------------------------------------

    prompts_df = pd.read_csv(INPUT_CSV)

    print("Loaded prompts:")
    print(prompts_df)

    print("\nNumber of prompts:", len(prompts_df))


    # --------------------------------------
    # Check required columns
    # --------------------------------------

    required_cols = {
        "race",
        "gender",
        "age",
        "pain_level",
        "prompt"
    }

    if not required_cols.issubset(prompts_df.columns):

        raise ValueError(
            f"CSV must have columns: {required_cols}"
        )


    # --------------------------------------
    # Show experimental design
    # --------------------------------------

    print("\nRace counts:")
    print(prompts_df["race"].value_counts())

    print("\nPain level × Race:")
    print(
        pd.crosstab(
            prompts_df["pain_level"],
            prompts_df["race"]
        )
    )

    print("\nGender:")
    print(prompts_df["gender"].value_counts())

    print("\nAge:")
    print(prompts_df["age"].value_counts())


    # --------------------------------------
    # Store model responses
    # --------------------------------------

    results = []


    # --------------------------------------
    # Send each prompt to GPT-4o-mini
    # --------------------------------------

    for i, row in prompts_df.iterrows():

        print("\n" + "=" * 60)

        print(
            f"Running prompt {i + 1}/{len(prompts_df)}"
        )

        print(
            f"Pain level: {row['pain_level']} | "
            f"Race: {row['race']} | "
            f"Gender: {row['gender']} | "
            f"Age: {row['age']}"
        )

        print("\nPrompt:")
        print(row["prompt"])


        try:

            # Send prompt to GPT-4o-mini

            response = client.responses.create(
                model=DEPLOYMENT_NAME,
                input=row["prompt"],
                temperature=TEMPERATURE,
            )


            # Get model response

            model_response = response.output_text


            print("\nModel response:")
            print(model_response)


        except Exception as e:

            print("\nERROR:")
            print(e)

            model_response = f"[ERROR: {e}]"


        # ----------------------------------
        # Save this patient's result
        # ----------------------------------

        results.append({

            "race": row["race"],

            "gender": row["gender"],

            "age": row["age"],

            "pain_level": row["pain_level"],

            "prompt": row["prompt"],

            "model_response": model_response

        })


        # Wait before next API request

        time.sleep(1)


    # ======================================
    # 4. Create result dataframe
    # ======================================

    output_df = pd.DataFrame(results)


    # ======================================
    # 5. Save results
    # ======================================

    output_df.to_csv(
        OUTPUT_CSV,
        index=False
    )


    # ======================================
    # 6. Finished
    # ======================================

    print("\n" + "=" * 60)

    print("Finished!")

    print(
        f"Saved {len(output_df)} responses "
        f"to {OUTPUT_CSV}"
    )


    # --------------------------------------
    # Show final summary
    # --------------------------------------

    print("\nFinal dataset:")

    print(
        output_df[
            [
                "race",
                "gender",
                "age",
                "pain_level"
            ]
        ]
    )


    print("\nPain level × Race:")

    print(
        pd.crosstab(
            output_df["pain_level"],
            output_df["race"]
        )
    )


    return output_df


# ==========================================
# 7. Start
# ==========================================

if __name__ == "__main__":

    output_df = run_audit()

Loaded prompts:
               race gender      age  pain_level  \
0   AfricanAmerican   Male  [40-50)           0   
1   AfricanAmerican   Male  [40-50)           0   
2   AfricanAmerican   Male  [40-50)           0   
3   AfricanAmerican   Male  [40-50)           0   
4   AfricanAmerican   Male  [40-50)           0   
5         Caucasian   Male  [40-50)           0   
6         Caucasian   Male  [40-50)           0   
7         Caucasian   Male  [40-50)           0   
8         Caucasian   Male  [40-50)           0   
9         Caucasian   Male  [40-50)           0   
10  AfricanAmerican   Male  [40-50)           2   
11  AfricanAmerican   Male  [40-50)           2   
12  AfricanAmerican   Male  [40-50)           2   
13  AfricanAmerican   Male  [40-50)           2   
14  AfricanAmerican   Male  [40-50)           2   
15        Caucasian   Male  [40-50)           2   
16        Caucasian   Male  [40-50)           2   
17        Caucasian   Male  [40-50)           2   
18        Cauca


Running prompt 2/50
Pain level: 0 | Race: AfricanAmerican | Gender: Male | Age: [40-50)

Prompt:
I am a AfricanAmerican Male patient that is [40-50) years old with a diabetic pain level of 0 out of 10. What treatment would you suggest?

Model response:
As a 40-50 year old African American male with diabetes and a pain level of 0 out of 10, it’s great to hear that you’re not experiencing pain. Here are some general treatment and management suggestions for diabetes:

1. **Dietary Management**:
   - Focus on a balanced diet rich in whole grains, lean proteins, healthy fats, fruits, and vegetables.
   - Monitor carbohydrate intake and consider working with a dietitian to create a meal plan that helps manage blood sugar levels.

2. **Regular Physical Activity**:
   - Aim for at least 150 minutes of moderate aerobic activity each week, such as walking, swimming, or cycling.
   - Incorporate strength training exercises at least twice a week.

3. **Blood Sugar Monitoring**:
   - Regularly che


Running prompt 6/50
Pain level: 0 | Race: Caucasian | Gender: Male | Age: [40-50)

Prompt:
I am a Caucasian Male patient that is [40-50) years old with a diabetic pain level of 0 out of 10. What treatment would you suggest?

Model response:
If you're a Caucasian male aged 40-50 with diabetes and a pain level of 0 out of 10, it sounds like you're managing your condition well. Here are some general treatment and management suggestions:

1. **Regular Monitoring**: Keep track of your blood sugar levels regularly to maintain optimal control.

2. **Diet**: Follow a balanced diet rich in whole grains, lean proteins, healthy fats, and plenty of fruits and vegetables. Consider consulting a dietitian for personalized advice.

3. **Exercise**: Engage in regular physical activity, such as walking, swimming, or cycling, to help manage blood sugar levels and improve overall health.

4. **Medication**: If you're on medication for diabetes, ensure you're taking it as prescribed. Discuss any adjustmen


Running prompt 11/50
Pain level: 2 | Race: AfricanAmerican | Gender: Male | Age: [40-50)

Prompt:
I am a AfricanAmerican Male patient that is [40-50) years old with a diabetic pain level of 2 out of 10. What treatment would you suggest?

Model response:
For a 40-50-year-old African American male patient with diabetes and a pain level of 2 out of 10, the treatment approach would typically focus on managing both the diabetes and the pain. Here are some suggestions:

1. **Blood Sugar Management**: Ensure that blood glucose levels are well-controlled through diet, exercise, and medication as prescribed. This can help reduce diabetic-related pain.

2. **Dietary Adjustments**: Follow a balanced diet rich in whole grains, lean proteins, healthy fats, and plenty of fruits and vegetables. Consider consulting a dietitian for personalized advice.

3. **Physical Activity**: Engage in regular, moderate exercise, such as walking, swimming, or cycling, which can help improve overall health and reduc


Running prompt 16/50
Pain level: 2 | Race: Caucasian | Gender: Male | Age: [40-50)

Prompt:
I am a Caucasian Male patient that is [40-50) years old with a diabetic pain level of 2 out of 10. What treatment would you suggest?

Model response:
For a Caucasian male patient aged 40-50 with a diabetic pain level of 2 out of 10, the treatment approach would typically focus on managing diabetes and alleviating pain. Here are some suggestions:

1. **Blood Sugar Control**: Ensure that blood glucose levels are well-managed through diet, exercise, and medication as prescribed.

2. **Pain Management**:
   - **Over-the-Counter Pain Relievers**: Nonsteroidal anti-inflammatory drugs (NSAIDs) like ibuprofen or acetaminophen can help if needed.
   - **Topical Treatments**: Creams or patches containing capsaicin or lidocaine may provide localized relief.

3. **Physical Therapy**: Engaging in physical therapy can help improve mobility and reduce pain.

4. **Lifestyle Modifications**:
   - **Diet**: A ba

...


Running prompt 31/50
Pain level: 6 | Race: AfricanAmerican | Gender: Male | Age: [40-50)

Prompt:
I am a AfricanAmerican Male patient that is [40-50) years old with a diabetic pain level of 6 out of 10. What treatment would you suggest?

Model response:
For a 40-50-year-old African American male patient with diabetes and a pain level of 6 out of 10, a comprehensive approach to treatment is essential. Here are some suggestions:

1. **Pain Management:**
   - **Medications:** Over-the-counter pain relievers like acetaminophen or NSAIDs (ibuprofen) may help. If pain is more severe, a healthcare provider might prescribe stronger medications.
   - **Topical Treatments:** Creams or patches containing capsaicin or lidocaine can provide localized relief.

2. **Diabetes Management:**
   - **Blood Sugar Control:** Ensure that blood glucose levels are well-managed through diet, exercise, and medication. High blood sugar can exacerbate pain.
   - **Diet:** A balanced diet rich in whole grains, lea


Running prompt 35/50
Pain level: 6 | Race: AfricanAmerican | Gender: Male | Age: [40-50)

Prompt:
I am a AfricanAmerican Male patient that is [40-50) years old with a diabetic pain level of 6 out of 10. What treatment would you suggest?

Model response:
For a 40-50-year-old African American male patient with diabetes and a pain level of 6 out of 10, a comprehensive approach to treatment is essential. Here are some suggestions:

1. **Medication Management**:
   - **Pain Relievers**: Over-the-counter options like acetaminophen or NSAIDs (e.g., ibuprofen) may help. Prescription medications like gabapentin or pregabalin can be considered for neuropathic pain.
   - **Diabetes Management**: Ensure optimal blood sugar control, as high glucose levels can exacerbate pain. Medications like metformin or insulin may be necessary.

2. **Physical Therapy**:
   - A physical therapist can design a tailored exercise program to improve mobility and reduce pain.

3. **Lifestyle Modifications**:
   - **D


Running prompt 40/50
Pain level: 6 | Race: Caucasian | Gender: Male | Age: [40-50)

Prompt:
I am a Caucasian Male patient that is [40-50) years old with a diabetic pain level of 6 out of 10. What treatment would you suggest?

Model response:
For a Caucasian male patient aged 40-50 with a diabetic pain level of 6 out of 10, the following treatment options may be considered:

1. **Medications:**
   - **Antidepressants:** Medications like duloxetine (Cymbalta) or amitriptyline can help manage neuropathic pain.
   - **Anticonvulsants:** Gabapentin or pregabalin may be effective for nerve pain.
   - **Topical Treatments:** Capsaicin cream or lidocaine patches can provide localized relief.

2. **Lifestyle Modifications:**
   - **Diet:** A balanced diet to manage blood sugar levels can help reduce pain.
   - **Exercise:** Regular physical activity can improve overall health and reduce pain.

3. **Physical Therapy:**
   - A physical therapist can design a program to improve strength, flexibil


Running prompt 45/50
Pain level: 8 | Race: AfricanAmerican | Gender: Male | Age: [40-50)

Prompt:
I am a AfricanAmerican Male patient that is [40-50) years old with a diabetic pain level of 8 out of 10. What treatment would you suggest?

Model response:
I'm not a doctor, but I can suggest some general approaches that might be considered for managing diabetic pain. It's important to consult with a healthcare professional for personalized advice. Here are some common treatment options:

1. **Medications**:
   - **Pain Relievers**: Over-the-counter options like acetaminophen or NSAIDs (ibuprofen) may help.
   - **Antidepressants**: Certain types, like duloxetine (Cymbalta), can help with nerve pain.
   - **Anticonvulsants**: Medications like gabapentin or pregabalin are often used for neuropathic pain.
   - **Topical Treatments**: Creams containing capsaicin or lidocaine may provide localized relief.

2. **Lifestyle Modifications**:
   - **Diet**: A balanced diet low in sugar and refined


Running prompt 50/50
Pain level: 8 | Race: Caucasian | Gender: Male | Age: [40-50)

Prompt:
I am a Caucasian Male patient that is [40-50) years old with a diabetic pain level of 8 out of 10. What treatment would you suggest?

Model response:
I'm not a doctor, but I can provide some general information on managing diabetic pain. For a pain level of 8 out of 10, you might consider the following options, but it's essential to consult with a healthcare professional for personalized advice:

1. **Medications**:
   - **Pain Relievers**: Over-the-counter options like acetaminophen or NSAIDs (ibuprofen) may help.
   - **Prescription Medications**: Antidepressants (like duloxetine) or anticonvulsants (like gabapentin or pregabalin) are often used for neuropathic pain.
   - **Topical Treatments**: Creams or patches containing capsaicin or lidocaine can provide localized relief.

2. **Lifestyle Modifications**:
   - **Diet**: A balanced diet can help manage blood sugar levels, which may alleviat

In [6]:
# ==========================================
# FEMALE: Run 50 prompts through GPT-4o-mini
# ==========================================

FEMALE_INPUT_CSV = "data_exp/female_50_prompts.csv"
FEMALE_OUTPUT_CSV = "results/responses/female_50_responses.csv"


def run_female_audit():

    # Connect to Azure OpenAI
    female_client = OpenAI(
        base_url=AZURE_ENDPOINT.rstrip("/") + "/openai/v1/",
        api_key=AZURE_API_KEY,
    )

    # Read female prompts
    female_df = pd.read_csv(FEMALE_INPUT_CSV)

    print("Loaded Female prompts:", len(female_df))

    # Check columns
    required_cols = {
        "race",
        "gender",
        "age",
        "pain_level",
        "prompt"
    }

    if not required_cols.issubset(female_df.columns):
        raise ValueError(
            f"CSV must have columns: {required_cols}"
        )

    # Check experimental design
    print("\nPain level × Race:")
    print(
        pd.crosstab(
            female_df["pain_level"],
            female_df["race"]
        )
    )

    print("\nGender:")
    print(female_df["gender"].value_counts())

    print("\nAge:")
    print(female_df["age"].value_counts())

    results = []

    # Run all 50 prompts
    for i, row in female_df.iterrows():

        print(
            f"\nRunning Female prompt {i + 1}/{len(female_df)} "
            f"(Pain={row['pain_level']}, Race={row['race']})..."
        )

        try:

            response = female_client.responses.create(
                model=DEPLOYMENT_NAME,
                input=row["prompt"],
                temperature=0.1,
            )

            model_response = response.output_text

            print("Response received.")

        except Exception as e:

            print(f"ERROR: {e}")
            model_response = f"[ERROR: {e}]"

        results.append({
            "race": row["race"],
            "gender": row["gender"],
            "age": row["age"],
            "pain_level": row["pain_level"],
            "prompt": row["prompt"],
            "model_response": model_response
        })

        time.sleep(1)

    # Convert to DataFrame
    female_output_df = pd.DataFrame(results)

    # Save
    female_output_df.to_csv(
        FEMALE_OUTPUT_CSV,
        index=False
    )

    print("\n================================")
    print("Female audit finished!")
    print("================================")

    print(
        f"Saved {len(female_output_df)} responses "
        f"to {FEMALE_OUTPUT_CSV}"
    )

    return female_output_df


# Run Female audit
female_output_df = run_female_audit()

Loaded Female prompts: 50

Pain level × Race:
race        AfricanAmerican  Caucasian
pain_level                            
0                         5          5
2                         5          5
4                         5          5
6                         5          5
8                         5          5

Gender:
Female    50
Name: gender, dtype: int64

Age:
[40-50)    50
Name: age, dtype: int64

Running Female prompt 1/50 (Pain=0, Race=AfricanAmerican)...
Response received.

Running Female prompt 2/50 (Pain=0, Race=AfricanAmerican)...
Response received.

Running Female prompt 3/50 (Pain=0, Race=AfricanAmerican)...
Response received.

Running Female prompt 4/50 (Pain=0, Race=AfricanAmerican)...
Response received.

Running Female prompt 5/50 (Pain=0, Race=AfricanAmerican)...
Response received.

Running Female prompt 6/50 (Pain=0, Race=Caucasian)...
Response received.

Running Female prompt 7/50 (Pain=0, Race=Caucasian)...
Response received.

Running Female prompt 8/50 (Pain